# 02 Generate

A teacher LLM (via LiteLLM) writes Hinglish/English questions per corpus paragraph: answerable ones with cited
answers, unanswerable ones the model must refuse, and a share of adversarial prompt-injection variants. Each row
gets its retrieved passages from the index. Input: `index/<version>/`, `data/splits.json`,
`configs/generation.yaml`. Output: `data/raw/{train,dev,test}.jsonl`.
Manifest: none yet; the stage resumes by skipping any `job_key` already present in `data/raw/`.

**Needs:** `TEACHER_MODEL` and the provider API key in the environment. CPU is fine.

In [ ]:
import json, os, sys
from collections import Counter
from pathlib import Path

# Run from the repo root so relative paths match the README runbook.
ROOT = Path.cwd()
while not (ROOT / "core").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PY = sys.executable  # `!{PY} -m ...` runs the CLI with this kernel's interpreter
print("repo root:", ROOT)

In [ ]:
INDEX = "index/rbi-2026-11-10"
RAW = Path("data/raw")
assert os.environ.get("TEACHER_MODEL"), "set TEACHER_MODEL (and the provider key) before running"

In [ ]:
!{PY} -m intelligence.generate --index {INDEX}

## Summary

In [ ]:
def read_jsonl(path):
    path = Path(path)
    return [json.loads(l) for l in path.open(encoding="utf-8")] if path.exists() else []


for f in sorted(RAW.glob("*.jsonl")):
    rows = read_jsonl(f)
    print(f"{f.name}: {len(rows)} rows", dict(Counter(r["kind"] for r in rows)), dict(Counter(r["lang"] for r in rows)))
for r in read_jsonl(RAW / "train.jsonl")[:3]:
    print(f"- [{r['kind']}/{r['lang']}] {r['question']}\n  -> {r['answer'][:120]}")